## Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.preprocessing import OneHotEncoder
import datetime
import h3
import geopandas
import json

In [32]:
df = pd.read_csv("../csv/Taxi_Trips_(2024-)_20260514.csv")

### 1.1 Converting data

Converting numerical data to numerical data type

In [33]:
df["Trip Miles"] = df["Trip Miles"].str.replace(".", "", regex=False)
df["Trip Miles"] = df["Trip Miles"].str.replace(",", ".", regex=False)
df["Trip Miles"] = pd.to_numeric(df["Trip Miles"])

In [34]:
df["Extras"] = df["Extras"].str.replace(".", "", regex=False)
df["Extras"] = df["Extras"].str.replace(",", ".", regex=False)
df["Extras"] = df["Extras"].str.replace("$", "", regex=False)
df["Extras"] = pd.to_numeric(df["Extras"])
df["Tips"] = df["Tips"].str.replace(".", "", regex=False)
df["Tips"] = df["Tips"].str.replace(",", ".", regex=False)
df["Tips"] = df["Tips"].str.replace("$", "", regex=False)
df["Tips"] = pd.to_numeric(df["Tips"])
df["Tolls"] = df["Tolls"].str.replace(".", "", regex=False)
df["Tolls"] = df["Tolls"].str.replace(",", ".", regex=False)
df["Tolls"] = df["Tolls"].str.replace("$", "", regex=False)
df["Tolls"] = pd.to_numeric(df["Tolls"])
df["Fare"] = df["Fare"].str.replace(".", "", regex=False)
df["Fare"] = df["Fare"].str.replace(",", ".", regex=False)
df["Fare"] = df["Fare"].str.replace("$", "", regex=False)
df["Fare"] = pd.to_numeric(df["Fare"])

In [35]:
df["Trip Total"] = df["Trip Total"].str.replace(".", "", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace(",", ".", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace("$", "", regex=False)
df["Trip Total"] = pd.to_numeric(df["Trip Total"])

Change Longitude and Latitude data to numerical data type

In [36]:
df["Pickup Centroid Latitude"] = df["Pickup Centroid Latitude"].str.replace(",", ".")
df["Pickup Centroid Latitude"] = pd.to_numeric(df["Pickup Centroid Latitude"])

df["Pickup Centroid Longitude"] = df["Pickup Centroid Longitude"].str.replace(",", ".")
df["Pickup Centroid Longitude"] = pd.to_numeric(df["Pickup Centroid Longitude"])

df["Dropoff Centroid Latitude"] = df["Dropoff Centroid Latitude"].str.replace(",", ".")
df["Dropoff Centroid Latitude"] = pd.to_numeric(df["Dropoff Centroid Latitude"])

df["Dropoff Centroid Longitude"] = df["Dropoff Centroid Longitude"].str.replace(",", ".")
df["Dropoff Centroid Longitude"] = pd.to_numeric(df["Dropoff Centroid Longitude"])

### 1.2 Cleaning data 

Delete all trips with time seconds < 60

In [37]:
df = df[df["Trip Miles"] >= 60]

Delete all trips with 0 total pay

In [38]:
df = df[df["Trip Total"] != 0]

Check if df has a location

In [39]:
df = df[df["Pickup Census Tract"].notnull() | df["Pickup Community Area"].notnull() | df["Pickup Centroid Location"].notnull()]

If Payment Type = Null chang to unknown

In [40]:
print(df["Payment Type"].unique())
df["Payment Type"] = df["Payment Type"].fillna("Unknown")

['Cash' 'No Charge' 'Credit Card' 'Dispute' 'Unknown' 'Mobile' 'Prcard']


## Preparing Data

One hot encode Payment Type

In [41]:
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[['Payment Type']])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(['Payment Type']),
    index=df.index
)
df = pd.concat([df, one_hot_df], axis=1)
df = df.drop('Payment Type', axis=1)

Converting location to hexagons

In [53]:
# there is no trip where Pickup Census Tract has data, where Pickup Centroid Location does not have
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Location"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Latitude"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Pickup Community Area"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Dropoff Community Area"].notnull() & df["Dropoff Centroid Longitude"].isnull()].count())

Trip ID                       0
Taxi ID                       0
Trip Start Timestamp          0
Trip End Timestamp            0
Trip Seconds                  0
Trip Miles                    0
Pickup Census Tract           0
Dropoff Census Tract          0
Pickup Community Area         0
Dropoff Community Area        0
Fare                          0
Tips                          0
Tolls                         0
Extras                        0
Trip Total                    0
Company                       0
Pickup Centroid Latitude      0
Pickup Centroid Longitude     0
Pickup Centroid Location      0
Dropoff Centroid Latitude     0
Dropoff Centroid Longitude    0
Dropoff Centroid  Location    0
Payment Type_Cash             0
Payment Type_Credit Card      0
Payment Type_Dispute          0
Payment Type_Mobile           0
Payment Type_No Charge        0
Payment Type_Prcard           0
Payment Type_Unknown          0
hex_loc_pick_up               0
hex_loc_drop_off              0
dtype: i

City boundry block for chicago?

In [43]:
#city_bounding_box = geopandas.read_file('')
#city_bounding_box_json_string = city_bounding_box.to_json()
#city_bounding_box_json = json.loads(city_bounding_box_json_string)
#city_bounding_box_poly = city_bounding_box_json["features"][0]

In [44]:
resolution = 7

In [ ]:

df["hex_loc_pick_up"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Pickup Centroid Latitude"],
        row["Pickup Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Pickup Centroid Latitude"]) and pd.notna(row["Pickup Centroid Longitude"])
    else 0,
    axis=1
)

df["hex_loc_drop_off"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Dropoff Centroid Latitude"],
        row["Dropoff Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Dropoff Centroid Latitude"]) and pd.notna(row["Dropoff Centroid Longitude"])
    else 0,
    axis=1
)

5506     87275934effffff
18947                  0
22260    87275934effffff
42089    872664d8affffff
48068                  0
Name: hex_loc_drop_off, dtype: object